This notebook demonstrates how to run the dashboard, streaming data directly from the DANDI Archive. It is recommended to run this notebook on the [DANDI Hub](hub.dandiarchive.org) using the "NWBStream" environment.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dandi/example-notebooks/blob/master/000055/BruntonLab/peterson21/dashboard.ipynb)

## Installing requirements

The cell below installs the Python packages needed to run this notebook, including `brunton-lab-to-nwb` from its GitHub repo (it is not on PyPI).

In [ ]:
# Expects Python 3.12 (Google Colab's default runtime as of 2026).
# Installs are scoped to the active interpreter via `uv pip install --system`,
# which is required because Colab runs the kernel outside a virtualenv.
# NOTE: this notebook uses `driver='ros3'` to stream NWB from S3 — Colab's
# prebuilt h5py includes ROS3, but pip-installed h5py on other platforms may not.
!pip install -q uv
!uv pip install --system \
    "pynwb>=2.8,<3" \
    "hdmf<4" \
    "dandi>=0.60" \
    "h5py>=3.10" \
    "numpy>=1.24" \
    "pandas>=2.0" \
    "matplotlib>=3.7" \
    "seaborn>=0.13" \
    "scipy>=1.11" \
    "tqdm>=4.66" \
    "ipywidgets>=8" \
    "natsort>=8" \
    "ndx-events<0.3" \
    "nilearn>=0.10" \
    "nwbwidgets>=0.11" \
    "zarr<3" \
    "ipython_genutils" \
    "bqplot>=0.12" \
    "plotly>=5" \
    "joblib>=1" \
    "git+https://github.com/catalystneuro/brunton-lab-to-nwb.git"

> **⚠️ Restart runtime after install**
>
> The install may upgrade packages already loaded in the kernel. Go to **Runtime → Restart session**, then **Run all cells below** (skip this install cell on re-run).

In [ ]:
from brunton_lab_to_nwb.brunton_widgets import BruntonDashboard
from dandi.dandiapi import DandiAPIClient
from pynwb import NWBHDF5IO

In [ ]:
with DandiAPIClient() as client:
    asset = client.get_dandiset("000055", "draft").get_asset_by_path(
        "sub-01/sub-01_ses-4_behavior+ecephys.nwb"
    )
    s3_path = asset.get_content_url(follow_redirects=1, strip_query=True)

io = NWBHDF5IO(s3_path, mode='r', load_namespaces=True, driver='ros3')
nwb = io.read()

In [ ]:
BruntonDashboard(nwb, tab1="stream")